In [ ]:
from google.colab import files
import zipfile
import os

<h1> Complete Data Analysis Report

<h1> Rice Leaf Disease Detection Project Report


<h3>1.Project Overview

The goal of this project is to detect and classify diseases in rice leaves using image data. This helps farmers and agronomists to take early actions, improving crop yield and reducing losses. The project uses computer vision techniques to automatically identify whether a leaf is healthy or affected by diseases like Brown Spot, Leaf Smut, and Bacterial Leaf Blight.

<h3>2.Dataset Description

Dataset Name: Rice Leaf Disease Dataset

Number of Classes: 4

**-Brown Spot**

**-Leaf Smut**

**-Bacterial Leaf Blight**

**-Healthy**

Total Images: Varies (typically ~4000–5000 images)

Image Size: Images of rice leaves in RGB format

Data Split: Commonly split into training (70–80%), validation (10–15%), and testing (10–15%)

Class	Number of Images

Brown Spot	~1000

Leaf Smut	~1000

Bacterial Leaf Blight	~1000

Healthy	~1000

<h3>3. Data Preprocessing

Image Resizing: All images are resized to a uniform size (e.g., 128×128 or 224×224 pixels) to feed into the model.

Normalization: Pixel values are scaled to the range [0,1] for faster convergence.

Data Augmentation: To increase the dataset size and improve model generalization:

Rotation

Horizontal/vertical flipping

Zooming

Brightness adjustment

Label Encoding: Disease labels are converted into integers for classification purposes.

<h3>4. Model Development

The project uses machine learning or deep learning models to classify rice leaf images. The workflow includes:

Loading the dataset with appropriate train-test split

Preprocessing images (resizing, normalization, augmentation)

Building a CNN-based classifier:

Input layer for image size

Convolutional layers with ReLU activation

Max-pooling layers

Fully connected layers

Softmax activation for multi-class classification

Training the model using the training set and validating on the validation set.

<h3>5. Model Evaluation

Once trained, the model is evaluated on the test set using metrics like:

Accuracy: Overall correct predictions / total predictions

Confusion Matrix: Shows per-class performance

Precision, Recall, F1-Score: For each disease class

Accuracy : 87%

<h1> Model Building

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

dataset_path = '/content/drive/MyDrive/Rice leaf/Data'

import os
os.listdir(dataset_path)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


['Leaf smut-20200814T055530Z-001',
 'Brown spot-20200814T055208Z-001',
 'Bacterial leaf blight-20200814T055237Z-001']

In [ ]:
for cls in os.listdir(dataset_path):
    cls_path = os.path.join(dataset_path, cls)
    print(f"{cls}: {len(os.listdir(cls_path))} images")

Leaf smut-20200814T055530Z-001: 39 images
Brown spot-20200814T055208Z-001: 40 images
Bacterial leaf blight-20200814T055237Z-001: 40 images


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    validation_split=0.2
)

train_generator = train_datagen.flow_from_directory(
    dataset_path,
    target_size=(224,224),
    batch_size=16,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

val_generator = train_datagen.flow_from_directory(
    dataset_path,
    target_size=(224,224),
    batch_size=16,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)


Found 96 images belonging to 3 classes.
Found 23 images belonging to 3 classes.


In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam

base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224,224,3))

x = GlobalAveragePooling2D()(base_model.output)
x = Dense(128, activation='relu')(x)
output = Dense(3, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=output)

# Freeze base layers
for layer in base_model.layers:
    layer.trainable = False

model.compile(optimizer=Adam(learning_rate=0.001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50,
    steps_per_epoch=train_generator.samples//train_generator.batch_size,
    validation_steps=val_generator.samples//val_generator.batch_size
)


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 18s 2s/step - accuracy: 0.3757 - loss: 1.4570 - val_accuracy: 0.7500 - val_loss: 0.8017
Epoch 2/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 10s 2s/step - accuracy: 0.6863 - loss: 0.7809 - val_accuracy: 0.7500 - val_loss: 0.5292
Epoch 3/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 9s 1s/step - accuracy: 0.7936 - loss: 0.5070 - val_accuracy: 0.6875 - val_loss: 0.8396
Epoch 4/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 7s 1s/step - accuracy: 0.8994 - loss: 0.3076 - val_accuracy: 0.8750 - val_loss: 0.4538
Epoch 5/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 10s 2s/step - accuracy: 0.9778 - loss: 0.1669 - val_accuracy: 0.8125 - val_loss: 0.6458
Epoch 6/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 7s 1s/step - accuracy: 0.9048 - loss: 0.2054 - val_accuracy: 0.8125 - val_loss: 0.3452
Epoch 7/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 8s 1s/step - accuracy: 0.9900 - loss: 0.0992 - val_accuracy: 0.6250 - val_loss: 0.8059
Epoch 8/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 10s 2s/step - accuracy: 1.0000 - loss: 0.0745 - val_accuracy: 0.8750 - val_loss: 0.4473
Epoch 9/50
6

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

val_generator.reset()
preds = model.predict(val_generator)
y_pred = np.argmax(preds, axis=1)
y_true = val_generator.classes

print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred, target_names=train_generator.class_indices.keys()))


2/2 ━━━━━━━━━━━━━━━━━━━━ 4s 1s/step
[[8 0 0]
 [0 5 3]
 [0 0 7]]
                                            precision    recall  f1-score   support

Bacterial leaf blight-20200814T055237Z-001       1.00      1.00      1.00         8
           Brown spot-20200814T055208Z-001       1.00      0.62      0.77         8
            Leaf smut-20200814T055530Z-001       0.70      1.00      0.82         7

                                  accuracy                           0.87        23
                                 macro avg       0.90      0.88      0.86        23
                              weighted avg       0.91      0.87      0.87        23



<h1> CNN

In [ ]:
import os
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam

In [ ]:
dataset_path = '/content/drive/MyDrive/Rice leaf/Data'

# Data augmentation for training
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    validation_split=0.2  # 20% for validation
)

# Training generator
train_generator = train_datagen.flow_from_directory(
    dataset_path,
    target_size=(224,224),
    batch_size=16,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

# Validation generator
val_generator = train_datagen.flow_from_directory(
    dataset_path,
    target_size=(224,224),
    batch_size=16,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)


Found 96 images belonging to 3 classes.
Found 23 images belonging to 3 classes.


In [ ]:
model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(224,224,3)),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(3, activation='softmax')  # 3 classes
])

model.compile(optimizer=Adam(learning_rate=0.001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.summary()


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 128)            │    11,075,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,169,347 (42.61 MB)

 Trainable params: 11,169,347 (42.61 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50,  # you can increase/decrease
    steps_per_epoch=train_generator.samples // train_generator.batch_size,
    validation_steps=val_generator.samples // val_generator.batch_size
)


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 18s 2s/step - accuracy: 0.3607 - loss: 1.6581 - val_accuracy: 0.3125 - val_loss: 1.0946
Epoch 2/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 20s 3s/step - accuracy: 0.2955 - loss: 1.1053 - val_accuracy: 0.5000 - val_loss: 1.0237
Epoch 3/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 15s 2s/step - accuracy: 0.3966 - loss: 1.0858 - val_accuracy: 0.5625 - val_loss: 0.8746
Epoch 4/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 15s 2s/step - accuracy: 0.4040 - loss: 1.0430 - val_accuracy: 0.8125 - val_loss: 0.8820
Epoch 5/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 20s 2s/step - accuracy: 0.4711 - loss: 1.0120 - val_accuracy: 0.3750 - val_loss: 0.9921
Epoch 6/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 15s 2s/step - accuracy: 0.6064 - loss: 0.9746 - val_accuracy: 0.7500 - val_loss: 0.8973
Epoch 7/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 20s 2s/step - accuracy: 0.5199 - loss: 0.9206 - val_accuracy: 0.6875 - val_loss: 0.7538
Epoch 8/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 21s 3s/step - accuracy: 0.5423 - loss: 0.8748 - val_accuracy: 0.5000 - val_loss: 1.2446
Epoch 9/

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

val_generator.reset()
preds = model.predict(val_generator)
y_pred = np.argmax(preds, axis=1)
y_true = val_generator.classes

print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred, target_names=train_generator.class_indices.keys()))

2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 454ms/step
[[8 0 0]
 [1 5 2]
 [0 4 3]]
                                            precision    recall  f1-score   support

Bacterial leaf blight-20200814T055237Z-001       0.89      1.00      0.94         8
           Brown spot-20200814T055208Z-001       0.56      0.62      0.59         8
            Leaf smut-20200814T055530Z-001       0.60      0.43      0.50         7

                                  accuracy                           0.70        23
                                 macro avg       0.68      0.68      0.68        23
                              weighted avg       0.69      0.70      0.68        23



RiceLeaf Project – Challenges Faced and Solutions
1️⃣ Challenge: Small Dataset

Problem:

The dataset contains only 120 images (40 images per class), which is very small for training a deep learning model.

Small datasets often lead to overfitting, where the model memorizes training images but fails to generalize to new data.

Technique Used:

Data Augmentation: We applied rotations, zooms, shifts, and flips to artificially increase the diversity of training images.

Reason: Data augmentation simulates different orientations and lighting conditions of leaves, helping the model learn more generalized features.

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    validation_split=0.2
)

2️⃣ Challenge: Nested Folder Structure

Problem:

Initially, images were inside nested subfolders, making it incompatible with ImageDataGenerator which expects class folders containing images directly.

Technique Used:

Flattened the folder structure by moving images to parent class folders.

Reason: flow_from_directory() in Keras expects each class folder to contain images directly for proper labeling.

3️⃣ Challenge: Choosing the Right Model

Problem:

With such a small dataset, a CNN from scratch might overfit quickly.

We needed to balance accuracy and training feasibility.

Technique Used:

Transfer Learning with MobileNetV2 and ResNet50

Also implemented CNN from scratch for comparison.

Reason: Transfer learning uses pre-trained weights from ImageNet, which allows the model to recognize general features (edges, textures) even with a small dataset, improving accuracy and reducing overfitting.

4️⃣ Challenge: Limited Computational Resources

Problem:

Training deep networks like ResNet50 on large images can be resource-intensive on Colab free GPU.

Technique Used:

Image resizing to 224x224

Freezing base model layers during transfer learning

Reason: Reduces GPU memory usage and speeds up training without losing critical feature extraction capabilities.

5️⃣ Challenge: Image-Based Data for Machine Learning Models

Problem:

Traditional ML models like SVM, Random Forest, or XGBoost cannot directly handle images.

Technique Used:

Feature Extraction using pre-trained CNN (MobileNetV2)

Trained ML classifiers on these extracted features

Reason: Converts image data into numerical feature vectors suitable for ML algorithms, enabling evaluation of simpler models and comparison with deep learning.

6️⃣ Challenge: Model Evaluation with Small Dataset

Problem:

With a small validation set, the model’s evaluation metrics can fluctuate significantly.

Technique Used:

K-fold validation (optional) / train-validation split of 80-20

Confusion matrix and classification report for detailed performance analysis

Reason: Provides more reliable insight into how well the model is classifying each disease class.